# Google Maps Review Data Preparation

Two-step pipeline:

1. **`googlemaps-review-prepare.csv`** — all 184k reviews flattened from 466 JSON files, with parsed approximate review date, reviewer info, aspect ratings, joined with hotel distance-to-coast metadata
2. **`googlemaps-review-en-vi.csv`** — filtered to English + Vietnamese with non-empty text, trimmed columns for topic modeling

**Prerequisites:** `data/hotel_with_distance.csv` must exist (run `pre-scraping/data-prepare.ipynb` first)

In [1]:
import json
import re
import pandas as pd
from pathlib import Path
from datetime import datetime
from dateutil.relativedelta import relativedelta
from tqdm.auto import tqdm

OUTPUTS_DIR   = Path("../../goorawling/outputs")
DATA_DIR      = Path("../../data")
OUT_DIR       = Path(".")

DISTANCE_PATH = DATA_DIR / "hotel_with_distance.csv"
PREPARE_OUT   = OUT_DIR  / "googlemaps-review-prepare.csv"
EN_VI_OUT     = OUT_DIR  / "googlemaps-review-en-vi.csv"

JSON_FILES = sorted(OUTPUTS_DIR.glob("hotel_*_reviews.json"))
print(f"JSON files found: {len(JSON_FILES)}")

JSON files found: 466


## 1. Parsing Helpers

In [2]:
# --- Rating time parser ---
_EDITED_PREFIX = 'Thời gian chỉnh sửa:'
_TIME_RE = re.compile(
    r'(một|\d+)\s*(giờ|ngày|tuần|tháng|năm)\s*trước',
    re.IGNORECASE
)

def parse_rating_time(raw: str, scrape_dt: datetime):
    """Convert Vietnamese relative time + scrape timestamp → (approx_datetime, is_edited).

    Returns (None, is_edited) when pattern is not recognised.
    """
    if not raw:
        return None, False

    edited = raw.strip().startswith(_EDITED_PREFIX)
    text   = raw.replace(_EDITED_PREFIX, '').strip()

    m = _TIME_RE.search(text)
    if not m:
        return None, edited

    qty_str, unit = m.group(1), m.group(2)
    qty = 1 if qty_str == 'một' else int(qty_str)

    delta = {
        'giờ':  relativedelta(hours=qty),
        'ngày': relativedelta(days=qty),
        'tuần': relativedelta(weeks=qty),
        'tháng':relativedelta(months=qty),
        'năm':  relativedelta(years=qty),
    }.get(unit)

    if delta is None:
        return None, edited

    return scrape_dt - delta, edited


# --- Rating parser: "5/5" → 5 ---
def parse_rating(raw: str):
    if not raw:
        return None
    m = re.match(r'(\d+)/', raw)
    return int(m.group(1)) if m else None


# --- Reviewer info parser ---
_REVIEWS_RE = re.compile(r'(\d+)\s*bài đánh giá')
_PHOTOS_RE  = re.compile(r'(\d+)\s*ảnh')

def parse_reviewer_info(raw: str):
    """Returns (is_local_guide, reviewer_reviews, reviewer_photos)."""
    if not raw:
        return False, None, None
    is_lg = 'Local Guide' in raw
    rev_m = _REVIEWS_RE.search(raw)
    pho_m = _PHOTOS_RE.search(raw)
    return (
        is_lg,
        int(rev_m.group(1)) if rev_m else None,
        int(pho_m.group(1)) if pho_m else None,
    )


# --- Aspect key mapping (Vietnamese → column name) ---
ASPECT_MAP = {
    'Phòng':                            'aspect_room',
    'Dịch vụ':                          'aspect_service',
    'Vị trí':                           'aspect_location',
    'Loại chuyến đi':                   'aspect_trip_type',
    'Nhóm khách du lịch':               'aspect_travel_group',
    'Điểm nổi bật của khách sạn':       'aspect_highlights',
    'Đồ ăn và thức uống':               'aspect_food_drink',
    'Hoạt động ở khu vực lân cận':      'aspect_nearby',
    'Nơi có thể đi bộ':                 'aspect_walkability',
    'Độ an toàn':                       'aspect_safety',
    'Những điểm đáng chú ý':            'aspect_notable',
}

print("Helpers defined.")

Helpers defined.


## 2. Flatten All JSON Files → Rows

In [3]:
_HOTEL_ID_RE = re.compile(r'hotel_(\d+)_reviews\.json$')

rows = []
skipped_files = []

for path in tqdm(JSON_FILES, desc="Parsing files"):
    id_m = _HOTEL_ID_RE.search(path.name)
    if not id_m:
        continue
    hotel_id = int(id_m.group(1))

    try:
        data = json.loads(path.read_text(encoding='utf-8'))
    except Exception as e:
        skipped_files.append((path.name, str(e)))
        continue

    meta         = data.get('metadata', {})
    scrape_ts_str= meta.get('timestamp', '')
    try:
        scrape_dt = datetime.strptime(scrape_ts_str, '%Y-%m-%d %H:%M:%S')
    except ValueError:
        scrape_dt = None

    for place_id, reviews in data.get('reviews_by_place', {}).items():
        for rv in reviews:
            ef   = rv.get('edge_fields', {})
            pn   = rv.get('place_node', {})
            rmeta= ef.get('metadata', {})
            rs   = ef.get('review_section', {})
            rt_  = rv.get('review_text', {})
            txt_block = rs.get('review_text', {})
            aspects   = rs.get('aspect_rating', {})
            respond   = rs.get('hotel_respond', {})

            rating_time_raw = rmeta.get('rating_time', '')
            approx_dt, is_edited = parse_rating_time(rating_time_raw, scrape_dt) \
                if scrape_dt else (None, False)

            is_lg, rev_count, photo_count = parse_reviewer_info(
                ef.get('reviewer_info', '')
            )

            review_text = txt_block.get('text', '') or ''

            row = {
                # Identity
                'hotel_id':             hotel_id,
                'review_id':            rmeta.get('review_id'),
                'place_id':             pn.get('place_id', place_id),
                # Scrape info
                'scrape_timestamp':     scrape_ts_str,
                # Reviewer
                'reviewer_name':        ef.get('reviewer_name'),
                'is_local_guide':       is_lg,
                'reviewer_reviews':     rev_count,
                'reviewer_photos':      photo_count,
                # Rating & timing
                'rating':               parse_rating(rmeta.get('rating', '')),
                'rating_time_raw':      rating_time_raw,
                'rating_time_edited':   is_edited,
                'approx_review_date':   approx_dt.date() if approx_dt else None,
                'review_year':          approx_dt.year  if approx_dt else None,
                'review_month':         approx_dt.month if approx_dt else None,
                'review_period':        approx_dt.strftime('%Y-%m') if approx_dt else None,
                # Review text
                'review_text':          review_text,
                'review_lang':          txt_block.get('lang', ''),
                'has_text':             bool(review_text.strip()),
                'image_count':          len(ef.get('image_urls', [])),
                'hotel_respond':        respond.get('text') if respond else None,
                # Aspects
                **{col: aspects.get(vi_key) for vi_key, col in ASPECT_MAP.items()},
            }
            rows.append(row)

print(f"\nTotal rows extracted : {len(rows):,}")
print(f"Files skipped        : {len(skipped_files)}")
if skipped_files:
    for name, err in skipped_files:
        print(f"  {name}: {err}")

Parsing files:   0%|          | 0/466 [00:00<?, ?it/s]


Total rows extracted : 184,962
Files skipped        : 0


In [4]:
df = pd.DataFrame(rows)
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head(3)

Shape: (184962, 31)
Columns: ['hotel_id', 'review_id', 'place_id', 'scrape_timestamp', 'reviewer_name', 'is_local_guide', 'reviewer_reviews', 'reviewer_photos', 'rating', 'rating_time_raw', 'rating_time_edited', 'approx_review_date', 'review_year', 'review_month', 'review_period', 'review_text', 'review_lang', 'has_text', 'image_count', 'hotel_respond', 'aspect_room', 'aspect_service', 'aspect_location', 'aspect_trip_type', 'aspect_travel_group', 'aspect_highlights', 'aspect_food_drink', 'aspect_nearby', 'aspect_walkability', 'aspect_safety', 'aspect_notable']


,hotel_id,review_id,place_id,scrape_timestamp,reviewer_name,is_local_guide,reviewer_reviews,reviewer_photos,rating,rating_time_raw,...,aspect_service,aspect_location,aspect_trip_type,aspect_travel_group,aspect_highlights,aspect_food_drink,aspect_nearby,aspect_walkability,aspect_safety,aspect_notable
0,1009828,Ci9DQUlRQUNvZENodHljRjlvT2xsQlVsOUlORXRwYW1oWG...,unknown,2026-03-24 13:55:08,AYLİN,True,11.0,6.0,4.0,một tháng trước trên,...,5,5,Chuyến nghỉ mát,Bạn bè,NaN,NaN,NaN,NaN,NaN,NaN
1,1009828,Ci9DQUlRQUNvZENodHljRjlvT2tOdWJERm5SVzF2VmpOVm...,unknown,2026-03-24 13:55:08,Văn tiến Hồ,True,2.0,2.0,5.0,2 tháng trước trên,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1009828,Ci9DQUlRQUNvZENodHljRjlvT25ScWIwTXhjVzVqUlhWRV...,unknown,2026-03-24 13:55:08,erika shimizu,True,21.0,15.0,5.0,2 tháng trước trên,...,5,5,Chuyến công tác,Cặp đôi,NaN,NaN,NaN,NaN,NaN,NaN


### Sanity checks

In [5]:
print("=== Review language distribution ===")
print(df['review_lang'].value_counts(dropna=False).head(10))

print("\n=== Rating distribution ===")
print(df['rating'].value_counts().sort_index())

print("\n=== has_text ===")
print(df['has_text'].value_counts())

print("\n=== approx_review_date null count ===")
print(f"Parsed: {df['approx_review_date'].notna().sum():,}  |  Failed: {df['approx_review_date'].isna().sum():,}")

print("\n=== review_period distribution (last 24) ===")
print(df['review_period'].value_counts().sort_index().tail(24))

print("\n=== Aspect fill rates ===")
aspect_cols = list(ASPECT_MAP.values())
print(df[aspect_cols].notna().sum().sort_values(ascending=False))

=== Review language distribution ===
review_lang
vi         78498
           57384
en         49004
ko             9
zh             9
fr             6
de             5
es             5
zh-Hant        4
ja             4
Name: count, dtype: int64

=== Rating distribution ===
rating
1.0      11235
2.0       5605
3.0      15555
4.0      34537
5.0     112946
6.0          2
8.0         15
9.0         13
10.0       111
Name: count, dtype: int64

=== has_text ===
has_text
True     127590
False     57372
Name: count, dtype: int64

=== approx_review_date null count ===
Parsed: 184,916  |  Failed: 46

=== review_period distribution (last 24) ===
review_period
2014-03      324
2015-03      579
2016-03     1225
2017-03     3180
2018-03     6247
2019-03     9775
2020-03    14567
2021-03     4933
2022-03     2587
2023-03    11611
2024-03    20191
2025-03    31198
2025-04     4189
2025-05     4126
2025-06     4474
2025-07     6080
2025-08     7106
2025-09     6712
2025-10     6799
2025-11     6984
202

## 3. Join Hotel Metadata + Distance to Coast

In [6]:
HOTEL_COLS = [
    'hotel_id', 'hotel_name', 'star_rating', 'city', 'state', 'country',
    'numberrooms', 'yearopened', 'yearrenovated', 'accommodation_type',
    'distance2coastline', 'hotel_coordinate', 'nearest_coordinate',
]

hotel_df = pd.read_csv(
    DISTANCE_PATH,
    encoding='utf-8-sig',
    usecols=HOTEL_COLS,
)
print(f"hotel_with_distance loaded: {hotel_df.shape}")
hotel_df.head(2)

hotel_with_distance loaded: (8574, 13)


,hotel_id,hotel_name,city,state,country,star_rating,numberrooms,yearopened,yearrenovated,accommodation_type,hotel_coordinate,distance2coastline,nearest_coordinate
0,163,Ramana Saigon Hotel,Ho Chi Minh City,NaN,Vietnam,4.0,296.0,1996.0,2015.0,Hotel,POINT (106.678101 10.787597),47.465,POINT (11907922.454514029 1165509.8719933222)
1,902,Lotte Hotel Saigon,Ho Chi Minh City,Ho Chi Minh,Vietnam,5.0,283.0,2001.0,2014.0,Hotel,POINT (106.706797 10.778689),44.573,POINT (11907922.454514029 1165509.8719933222)


In [7]:
df = df.merge(hotel_df, on='hotel_id', how='left')

missing_dist = df['distance2coastline'].isna().sum()
print(f"Shape after merge        : {df.shape}")
print(f"Reviews with distance    : {len(df) - missing_dist:,} / {len(df):,}")
print(f"Missing distance2coast   : {missing_dist:,}")

df[['hotel_id', 'hotel_name', 'distance2coastline', 'star_rating', 'review_period']].head(5)

Shape after merge        : (184962, 43)
Reviews with distance    : 184,962 / 184,962
Missing distance2coast   : 0


,hotel_id,hotel_name,distance2coastline,star_rating,review_period
0,1009828,Ramada Encore by Wyndham Saigon D1 - Formerly ...,44.6,4.0,2026-02
1,1009828,Ramada Encore by Wyndham Saigon D1 - Formerly ...,44.6,4.0,2026-01
2,1009828,Ramada Encore by Wyndham Saigon D1 - Formerly ...,44.6,4.0,2026-01
3,1009828,Ramada Encore by Wyndham Saigon D1 - Formerly ...,44.6,4.0,2026-01
4,1009828,Ramada Encore by Wyndham Saigon D1 - Formerly ...,44.6,4.0,2025-12


## 4. Save `googlemaps-review-prepare.csv`

Full enriched dataset — all languages, all columns.

In [8]:
df.to_csv(PREPARE_OUT, index=False, encoding='utf-8-sig')
print(f"Saved → {PREPARE_OUT}")
print(f"Shape : {df.shape}")
print(f"Columns ({len(df.columns)}): {df.columns.tolist()}")

Saved → googlemaps-review-prepare.csv
Shape : (184962, 43)
Columns (43): ['hotel_id', 'review_id', 'place_id', 'scrape_timestamp', 'reviewer_name', 'is_local_guide', 'reviewer_reviews', 'reviewer_photos', 'rating', 'rating_time_raw', 'rating_time_edited', 'approx_review_date', 'review_year', 'review_month', 'review_period', 'review_text', 'review_lang', 'has_text', 'image_count', 'hotel_respond', 'aspect_room', 'aspect_service', 'aspect_location', 'aspect_trip_type', 'aspect_travel_group', 'aspect_highlights', 'aspect_food_drink', 'aspect_nearby', 'aspect_walkability', 'aspect_safety', 'aspect_notable', 'hotel_name', 'city', 'state', 'country', 'star_rating', 'numberrooms', 'yearopened', 'yearrenovated', 'accommodation_type', 'hotel_coordinate', 'distance2coastline', 'nearest_coordinate']


## 5. Filter to English + Vietnamese

Keep `review_lang in ['en', 'vi']` AND `has_text == True`, select topic-modeling columns.

In [9]:
en_vi = df[
    df['review_lang'].isin(['en', 'vi']) & df['has_text']
].copy()

print(f"All reviews              : {len(df):,}")
print(f"en + vi with text        : {len(en_vi):,}")
print()
print("Language breakdown:")
print(en_vi['review_lang'].value_counts())

All reviews              : 184,962
en + vi with text        : 127,502

Language breakdown:
review_lang
vi    78498
en    49004
Name: count, dtype: int64


In [10]:
TOPIC_COLS = [
    # Primary text
    'review_text',
    'review_lang',
    # Rating
    'rating',
    # Temporal
    'review_year',
    'review_month',
    'review_period',
    'approx_review_date',
    'rating_time_edited',
    # Reviewer
    'is_local_guide',
    'reviewer_reviews',
    'reviewer_photos',
    # Aspects — sub-ratings & segmentation
    'aspect_room',
    'aspect_service',
    'aspect_location',
    'aspect_trip_type',
    'aspect_travel_group',
    'aspect_highlights',
    'aspect_food_drink',
    # Hotel identity
    'hotel_id',
    'hotel_name',
    'city',
    'state',
    # Hotel attributes
    'star_rating',
    'accommodation_type',
    'numberrooms',
    'yearopened',
    'yearrenovated',
    # Coastal proximity
    'distance2coastline',
    'hotel_coordinate',
    # Review metadata
    'review_id',
    'image_count',
]

available = [c for c in TOPIC_COLS if c in en_vi.columns]
missing   = [c for c in TOPIC_COLS if c not in en_vi.columns]
if missing:
    print("Warning — columns not found (skipped):", missing)

en_vi = en_vi[available].reset_index(drop=True)
print(f"Final shape: {en_vi.shape}")
en_vi.head(3)

Final shape: (127502, 31)


,review_text,review_lang,rating,review_year,review_month,review_period,approx_review_date,rating_time_edited,is_local_guide,reviewer_reviews,...,state,star_rating,accommodation_type,numberrooms,yearopened,yearrenovated,distance2coastline,hotel_coordinate,review_id,image_count
0,Các phòng khá nhỏ.,vi,4.0,2026.0,2.0,2026-02,2026-02-24,False,True,11.0,...,Ho Chi Minh,4.0,Hotel,70.0,2015.0,2022.0,44.6,POINT (106.705774 10.77812),Ci9DQUlRQUNvZENodHljRjlvT2xsQlVsOUlORXRwYW1oWG...,0
1,Thủ tục nhận phòng diễn ra suôn sẻ hơn nhiều s...,vi,5.0,2026.0,1.0,2026-01,2026-01-24,False,True,21.0,...,Ho Chi Minh,4.0,Hotel,70.0,2015.0,2022.0,44.6,POINT (106.705774 10.77812),Ci9DQUlRQUNvZENodHljRjlvT25ScWIwTXhjVzVqUlhWRV...,2
2,"Great location for a clean, quiet hotel, tucke...",en,5.0,2026.0,1.0,2026-01,2026-01-24,False,True,65.0,...,Ho Chi Minh,4.0,Hotel,70.0,2015.0,2022.0,44.6,POINT (106.705774 10.77812),Ci9DQUlRQUNvZENodHljRjlvT21aa1prOU5hVkJVTlVsdV...,4


In [11]:
print("=== Null audit on key columns ===")
print(en_vi[['review_text', 'rating', 'review_period', 'distance2coastline',
             'aspect_room', 'aspect_service', 'aspect_location',
             'aspect_trip_type', 'aspect_travel_group']].isna().sum())

print("\n=== review_period distribution (last 24) ===")
print(en_vi['review_period'].value_counts().sort_index().tail(24))

print("\n=== Rating distribution ===")
print(en_vi['rating'].value_counts().sort_index())

print("\n=== Trip type distribution ===")
print(en_vi['aspect_trip_type'].value_counts().head(10))

print("\n=== Travel group distribution ===")
print(en_vi['aspect_travel_group'].value_counts().head(10))

=== Null audit on key columns ===
review_text                 0
rating                   4933
review_period              33
distance2coastline          0
aspect_room             85203
aspect_service          88212
aspect_location         90784
aspect_trip_type       103044
aspect_travel_group    106415
dtype: int64

=== review_period distribution (last 24) ===
review_period
2014-03      295
2015-03      542
2016-03     1098
2017-03     2506
2018-03     4073
2019-03     5854
2020-03     8318
2021-03     2357
2022-03     1445
2023-03     7601
2024-03    13455
2025-03    22131
2025-04     3040
2025-05     2901
2025-06     3195
2025-07     4210
2025-08     5084
2025-09     4968
2025-10     5074
2025-11     5102
2025-12     6187
2026-01     7012
2026-02     5344
2026-03     5402
Name: count, dtype: int64

=== Rating distribution ===
rating
1.0      9306
2.0      4171
3.0      9429
4.0     20823
5.0     78708
6.0         2
8.0        14
9.0        10
10.0      106
Name: count, dtype: int64



## 6. Save `googlemaps-review-en-vi.csv`

Topic modeling ready — English + Vietnamese only, non-empty text, trimmed columns.

In [12]:
en_vi.to_csv(EN_VI_OUT, index=False, encoding='utf-8-sig')
print(f"Saved → {EN_VI_OUT}")
print(f"Shape : {en_vi.shape}")
print(f"Columns: {en_vi.columns.tolist()}")

Saved → googlemaps-review-en-vi.csv
Shape : (127502, 31)
Columns: ['review_text', 'review_lang', 'rating', 'review_year', 'review_month', 'review_period', 'approx_review_date', 'rating_time_edited', 'is_local_guide', 'reviewer_reviews', 'reviewer_photos', 'aspect_room', 'aspect_service', 'aspect_location', 'aspect_trip_type', 'aspect_travel_group', 'aspect_highlights', 'aspect_food_drink', 'hotel_id', 'hotel_name', 'city', 'state', 'star_rating', 'accommodation_type', 'numberrooms', 'yearopened', 'yearrenovated', 'distance2coastline', 'hotel_coordinate', 'review_id', 'image_count']
